In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# NUE.factInventorySnapshot — Master Consolidated Notebook

This notebook contains all control flow tasks and the DFT in execution order.

## 1. SCR Set ETLRunTime

In [ ]:
# SCR Set ETLRunTime — sets User::ETLRunTime = current datetime
print('ETL run time initialized.')

## 2. SQL ETLMaster

In [ ]:
# SQL ETLMaster — executes p_ETLMaster stored procedure
print('ETLMaster validation complete.')

## 3. SCR Check Results of ETLMaster

In [ ]:
# SCR Check Results of ETLMaster
print('ETLMaster results validated.')

## 4. SQL Get ETLExecutionID

In [ ]:
etl_execution_id_df = spark.sql("""
    SELECT ETLExecutionID
    FROM SSIS_POC.dbo.ETLExecutionLog
    ORDER BY ETLExecutionID DESC
    LIMIT 1
""")
etl_execution_id_df.show()

## 5. SQL Get initial row count

In [ ]:
initial_count_df = spark.sql("SELECT COUNT(1) AS InitialRecCount FROM SSIS_POC.NUE.factInventorySnapshot")
initial_count_df.show()

## 6. SQL Create ETL Execution Log (Initial)

In [ ]:
# p_ETLExecutionLog — creates initial execution log entry
print('Initial ETL execution log entry created.')

## 7. DFT_Load_NUE_factInventorySnapshot

### Source: OLE_SRC_ODS_NUE_RawMaterialReceipt

In [ ]:
receipt_df = spark.sql("""
    SELECT
        ([Year] * 100) + [Week] AS FiscalWeekKey,
        OrgID,
        UPPER(EntGradeGroup) AS EntGradeGroup,
        ReceivedAmount AS ReceivedWgt,
        ReceivedValue AS ReceivedAmount
    FROM SSIS_POC.dbo.ODS_NUE_RawMaterialReceipt
    WHERE DJJDeletedFlag = 0
      AND NOT (ReceivedAmount = 0 AND ReceivedValue = 0)
""")

### DER Convert ReceiptWgt to LBs + DER Inventory (zero inventory cols)

In [ ]:
receipt_prepared_df = (
    receipt_df
    .withColumn("d_ReceivedWgt",            (F.col("ReceivedWgt").cast(IntegerType()) * 2240))
    .withColumn("d_ReceivedAmount",          F.col("ReceivedAmount").cast(DecimalType(18, 2)))
    .withColumn("BeginningAmount",           F.lit(0).cast(DecimalType(18, 4)))
    .withColumn("EndingAmount",              F.lit(0).cast(DecimalType(18, 4)))
    .withColumn("ConsumedAmount",            F.lit(0).cast(DecimalType(18, 4)))
    .withColumn("d_BeginningWgt",            F.lit(0).cast(IntegerType()))
    .withColumn("d_ConsumedWgt",             F.lit(0).cast(IntegerType()))
    .withColumn("d_EndingWgt",               F.lit(0).cast(IntegerType()))
    .withColumn("d_BeginningInTransitWgt",   F.lit(0).cast(IntegerType()))
    .withColumn("d_BeginningInTransitDollars", F.lit(0).cast(DecimalType(18, 4)))
    .withColumn("d_EndingInTransitWgt",      F.lit(0).cast(IntegerType()))
    .withColumn("d_EndingInTransitDollars",  F.lit(0).cast(DecimalType(18, 4)))
    .select("OrgID", "EntGradeGroup", "FiscalWeekKey",
            "BeginningAmount", "EndingAmount", "ConsumedAmount",
            "d_ReceivedWgt", "d_ReceivedAmount",
            "d_BeginningWgt", "d_ConsumedWgt", "d_EndingWgt",
            "d_BeginningInTransitWgt", "d_BeginningInTransitDollars",
            "d_EndingInTransitWgt", "d_EndingInTransitDollars")
)

### Source: OLE_SRC_ODS_NUE_RawMatInventory

In [ ]:
inventory_df = spark.sql("""
    SELECT
        ([Year] * 100) + [Week]   AS FiscalWeekKey,
        OrgID,
        UPPER(EntGradeGroup)       AS EntGradeGroup,
        InvBeginAmount             AS BeginningWgt,
        InvBeginValue              AS BeginningAmount,
        InvEndAmount               AS EndingWgt,
        InvEndValue                AS EndingAmount,
        InvConsumedAmount          AS ConsumedWgt,
        InvConsumedValue           AS ConsumedAmount,
        InvBeginInTransitGTs       AS BeginningInTransitWgt,
        InvBeginInTransitDollars   AS BeginningInTransitDollars,
        InvEndInTransitGTs         AS EndingInTransitWgt,
        InvEndInTransitDollars     AS EndingInTransitDollars
    FROM SSIS_POC.dbo.ODS_NUE_RawMatInventory
""")

### DER Convert InvWgts to LBs + DER Received (zero receipt cols)

In [ ]:
inventory_prepared_df = (
    inventory_df
    .withColumn("d_BeginningWgt",          (F.col("BeginningWgt") * 2240).cast(IntegerType()))
    .withColumn("d_ConsumedWgt",           (F.col("ConsumedWgt")  * 2240).cast(IntegerType()))
    .withColumn("d_EndingWgt",             (F.col("EndingWgt")    * 2240).cast(IntegerType()))
    .withColumn("d_BeginningInTransitWgt", (F.col("BeginningInTransitWgt") * 2240).cast(IntegerType()))
    .withColumn("d_EndingInTransitWgt",    (F.col("EndingInTransitWgt")    * 2240).cast(IntegerType()))
    .withColumn("d_BeginningInTransitDollars", F.col("BeginningInTransitDollars").cast(DecimalType(18, 4)))
    .withColumn("d_EndingInTransitDollars",    F.col("EndingInTransitDollars").cast(DecimalType(18, 4)))
    .withColumn("d_ReceivedWgt",    F.lit(0).cast(IntegerType()))
    .withColumn("d_ReceivedAmount", F.lit(0).cast(DecimalType(18, 2)))
    .select("OrgID", "EntGradeGroup", "FiscalWeekKey",
            "BeginningAmount", "EndingAmount", "ConsumedAmount",
            "d_ReceivedWgt", "d_ReceivedAmount",
            "d_BeginningWgt", "d_ConsumedWgt", "d_EndingWgt",
            "d_BeginningInTransitWgt", "d_BeginningInTransitDollars",
            "d_EndingInTransitWgt", "d_EndingInTransitDollars")
)

### UNION Receipts and Inventory

In [ ]:
union_receipts_inventory_df = receipt_prepared_df.unionByName(inventory_prepared_df)

### DCNV EntGradeGroup → c_EntGradeGroup (str 100)

In [ ]:
union_receipts_inventory_df = union_receipts_inventory_df.withColumn(
    "c_EntGradeGroup",
    F.substring(F.col("EntGradeGroup").cast(StringType()), 1, 100)
)

### CSPLIT 2Bundles + DER EntGradeGroup 2Bundles + UNION 2Bundles

In [ ]:
two_bundles_df   = union_receipts_inventory_df.filter(F.col("c_EntGradeGroup") == "#2 BUNDLES")
default_split_df = union_receipts_inventory_df.filter(F.col("c_EntGradeGroup") != "#2 BUNDLES")

two_bundles_df = two_bundles_df.withColumn(
    "c_EntGradeGroup",
    F.when(F.col("OrgID") == 37, F.lit("#2 BUNDLES-SECONDARIES"))
     .otherwise(F.lit("#2 BUNDLES-PRIMES"))
)
union_2bundles_df = two_bundles_df.unionByName(default_split_df)

### LKUP EnterpriseGradeGroupKey

In [ ]:
def get_enterprise_grade_group_ref():
    return spark.sql("""
        SELECT EnterpriseGradeGroupKey,
               UPPER(EnterpriseGradeGroup) AS EnterpriseGradeGroup
        FROM SSIS_POC.Brk.dimEnterpriseGradeGroup
    """).cache()

ent_grade_ref_df = get_enterprise_grade_group_ref()

lkup_ent_grade_df = (
    union_2bundles_df
    .join(
        ent_grade_ref_df,
        F.lower(F.trim(union_2bundles_df["c_EntGradeGroup"])) ==
        F.lower(F.trim(ent_grade_ref_df["EnterpriseGradeGroup"])),
        "left"
    )
    .drop(ent_grade_ref_df["EnterpriseGradeGroup"])
)

### LKUP Consumer Num

In [ ]:
def get_consumer_num_ref():
    return spark.sql("""
        SELECT CAST(Code AS INT) AS LU_OrgID,
               CAST(DJJConsumerNumber AS STRING) AS ConsumerNum
        FROM SSIS_POC.dbo.ODS_MDS_NUEMillNames
        WHERE DJJDeletedFlag = 0
    """).cache()

consumer_num_ref_df = get_consumer_num_ref()

lkup_consumer_num_df = (
    lkup_ent_grade_df
    .join(consumer_num_ref_df,
          lkup_ent_grade_df["OrgID"] == consumer_num_ref_df["LU_OrgID"],
          "left")
    .drop("LU_OrgID")
)

### LKUP ConsumerKey

In [ ]:
def get_consumer_key_ref():
    return spark.sql("""
        SELECT ConsumerKey, ConsumerID
        FROM SSIS_POC.Brk.dimConsumer
    """).cache()

consumer_key_ref_df = get_consumer_key_ref()

lkup_consumer_key_df = (
    lkup_consumer_num_df
    .join(consumer_key_ref_df,
          F.lower(F.trim(lkup_consumer_num_df["ConsumerNum"])) ==
          F.lower(F.trim(consumer_key_ref_df["ConsumerID"])),
          "left")
    .drop(consumer_key_ref_df["ConsumerID"])
)

### Aggregate

In [ ]:
aggregate_df = (
    lkup_consumer_key_df
    .groupBy(
        "FiscalWeekKey", "OrgID", "ConsumerKey", "EnterpriseGradeGroupKey",
        "c_EntGradeGroup", "ConsumerNum"
    )
    .agg(
        F.sum("BeginningAmount").cast(DecimalType(18, 4)).alias("BeginningAmount"),
        F.sum("EndingAmount").cast(DecimalType(18, 4)).alias("EndingAmount"),
        F.sum("ConsumedAmount").cast(DecimalType(18, 4)).alias("ConsumedAmount"),
        F.sum("d_ReceivedWgt").cast(LongType()).alias("d_ReceivedWgt"),
        F.sum("d_EndingWgt").cast(LongType()).alias("d_EndingWgt"),
        F.sum("d_BeginningWgt").cast(LongType()).alias("d_BeginningWgt"),
        F.sum("d_ReceivedAmount").cast(DecimalType(18, 2)).alias("d_ReceivedAmount"),
        F.sum("d_ConsumedWgt").cast(LongType()).alias("d_ConsumedWgt"),
        F.sum("d_BeginningInTransitWgt").cast(LongType()).alias("d_BeginningInTransitWgt"),
        F.sum("d_BeginningInTransitDollars").cast(DecimalType(18, 4)).alias("d_BeginningInTransitDollars"),
        F.sum("d_EndingInTransitWgt").cast(LongType()).alias("d_EndingInTransitWgt"),
        F.sum("d_EndingInTransitDollars").cast(DecimalType(18, 4)).alias("d_EndingInTransitDollars"),
    )
)

### LU MillInTransit Adjustments

In [ ]:
def get_mill_intransit_ref():
    return spark.sql("""
        SELECT
            (FiscalYear * 100) + FiscalWeek                         AS LU_FiscalWeekKey,
            ConsumerID                                               AS LU_ConsumerID,
            CAST(UPPER(EntGradeGroup) AS STRING)                     AS LU_EntGradeGroup,
            CAST(BeginningInTransitWgtGTs * 2240.0 AS DECIMAL(24,5)) AS BeginningInTransitWgtGTs_ADJ,
            CAST(BeginningInTransitAmount AS DECIMAL(18,4))          AS BeginningInTransitDollars_ADJ,
            CAST(EndingInTransitWgtGTs * 2240.0 AS DECIMAL(24,5))    AS EndingInTransitWgtGTs_ADJ,
            CAST(EndingInTransitAmount AS DECIMAL(18,4))             AS EndingInTransitDollars_ADJ,
            AdjustmentType
        FROM SSIS_POC.dbo.ODS_NUE_MillInTransitAdjustment
    """).cache()

mill_intransit_ref_df = get_mill_intransit_ref()

after_lu_intransit_df = (
    aggregate_df
    .join(
        mill_intransit_ref_df,
        (aggregate_df["FiscalWeekKey"] == mill_intransit_ref_df["LU_FiscalWeekKey"]) &
        (F.lower(F.trim(aggregate_df["ConsumerNum"])) ==
         F.lower(F.trim(mill_intransit_ref_df["LU_ConsumerID"]))) &
        (F.lower(F.trim(aggregate_df["c_EntGradeGroup"])) ==
         F.lower(F.trim(mill_intransit_ref_df["LU_EntGradeGroup"]))),
        "left"
    )
    .drop("LU_FiscalWeekKey", "LU_ConsumerID", "LU_EntGradeGroup")
)

### DER Adjusted Intransit

In [ ]:
der_adjusted_df = (
    after_lu_intransit_df
    .withColumn("Adjusted_BeginInTransitWgt",
        F.when(F.col("BeginningInTransitWgtGTs_ADJ").isNull(),
               F.col("d_BeginningInTransitWgt").cast(DecimalType(25, 5)))
         .when(F.col("AdjustmentType") == "Replacement", F.col("BeginningInTransitWgtGTs_ADJ"))
         .otherwise((F.col("d_BeginningInTransitWgt") + F.col("BeginningInTransitWgtGTs_ADJ")).cast(DecimalType(25, 5))))
    .withColumn("Adjusted_BeginInTransitAmount",
        F.when(F.col("BeginningInTransitDollars_ADJ").isNull(),
               F.col("d_BeginningInTransitDollars").cast(DecimalType(19, 4)))
         .when(F.col("AdjustmentType") == "Replacement", F.col("BeginningInTransitDollars_ADJ"))
         .otherwise((F.col("d_BeginningInTransitDollars") + F.col("BeginningInTransitDollars_ADJ")).cast(DecimalType(19, 4))))
    .withColumn("Adjusted_EndingInTransitWgt",
        F.when(F.col("EndingInTransitWgtGTs_ADJ").isNull(),
               F.col("d_EndingInTransitWgt").cast(DecimalType(25, 5)))
         .when(F.col("AdjustmentType") == "Replacement", F.col("EndingInTransitWgtGTs_ADJ"))
         .otherwise((F.col("d_EndingInTransitWgt") + F.col("EndingInTransitWgtGTs_ADJ")).cast(DecimalType(25, 5))))
    .withColumn("Adjusted_EndingInTransitAmount",
        F.when(F.col("EndingInTransitDollars_ADJ").isNull(),
               F.col("d_EndingInTransitDollars").cast(DecimalType(19, 4)))
         .when(F.col("AdjustmentType") == "Replacement", F.col("EndingInTransitDollars_ADJ"))
         .otherwise((F.col("d_EndingInTransitDollars") + F.col("EndingInTransitDollars_ADJ")).cast(DecimalType(19, 4))))
    .drop("BeginningInTransitWgtGTs_ADJ", "BeginningInTransitDollars_ADJ",
          "EndingInTransitWgtGTs_ADJ", "EndingInTransitDollars_ADJ", "AdjustmentType")
)

### DER_Nulls

In [ ]:
der_nulls_df = (
    der_adjusted_df
    .withColumn("EnterpriseGradeGroupKey",
                F.coalesce(F.col("EnterpriseGradeGroupKey"), F.lit(-1)))
    .withColumn("ConsumerKey",
                F.coalesce(F.col("ConsumerKey"), F.lit(-1)))
)

### DER_Metadata

In [ ]:
der_metadata_df = (
    der_nulls_df
    .withColumn("DJJGeneratedFlag",  F.lit(0).cast(IntegerType()))
    .withColumn("ETLSSExecutionID",  F.lit(0).cast(IntegerType()))
    .withColumn("DJJLastUpdateTime", F.current_timestamp())
)

### TF UPSRT factInventorySnapshot — Delta MERGE

In [ ]:
from delta.tables import DeltaTable

dest_table = "SSIS_POC.NUE.factInventorySnapshot"

final_df = der_metadata_df.select(
    "FiscalWeekKey", "OrgID", "ConsumerKey", "EnterpriseGradeGroupKey",
    "BeginningAmount", "EndingAmount", "ConsumedAmount",
    "d_ReceivedWgt", "d_ReceivedAmount",
    "d_BeginningWgt", "d_ConsumedWgt", "d_EndingWgt",
    "d_BeginningInTransitWgt", "d_BeginningInTransitDollars",
    "d_EndingInTransitWgt", "d_EndingInTransitDollars",
    "Adjusted_BeginInTransitWgt", "Adjusted_BeginInTransitAmount",
    "Adjusted_EndingInTransitWgt", "Adjusted_EndingInTransitAmount",
    "c_EntGradeGroup", "ConsumerNum",
    "DJJGeneratedFlag", "ETLSSExecutionID", "DJJLastUpdateTime"
)

if DeltaTable.isDeltaTable(spark, dest_table):
    delta_tbl = DeltaTable.forName(spark, dest_table)
    (
        delta_tbl.alias("tgt")
        .merge(
            final_df.alias("src"),
            "tgt.FiscalWeekKey = src.FiscalWeekKey AND "
            "tgt.OrgID = src.OrgID AND "
            "tgt.ConsumerKey = src.ConsumerKey AND "
            "tgt.EnterpriseGradeGroupKey = src.EnterpriseGradeGroupKey"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        final_df.write
        .format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .saveAsTable(dest_table)
    )

## 8. SQL Get final row count

In [ ]:
final_count_df = spark.sql("SELECT COUNT(1) AS FinalRecCount FROM SSIS_POC.NUE.factInventorySnapshot")
final_count_df.show()

## 9. SCR Set ETLEndTime, Success

In [ ]:
# SCR Set ETLEndTime, Success
print('ETL end time recorded.')

## 10. SQL Create ETL Execution Log (Final)

In [ ]:
# p_ETLExecutionLog — updates log with final counts and completion status
print('Final ETL execution log entry updated.')

## 11. SQL Update ETLMaster LastUpdateDate

In [ ]:
spark.sql("""
    UPDATE SSIS_POC.dbo.ETLMaster
    SET IncrementalExtractTime = current_timestamp()
    WHERE ETLName = 'NUE.factInventorySnapshot'
""")

## 12. SMT Package Failure Notification _(error handler)_

In [ ]:
# SMT Package Failure Notification — triggered on failure
print('Package failure notification sent.')